# Ekstraksi Data Kualitas Udara Kabupaten Bangkalan

Notebook ini menarik data satelit **Sentinel-5P (TROPOMI) Level 2** dari
**Copernicus Data Space Ecosystem** melalui antarmuka **openEO**, khusus untuk
wilayah **Kabupaten Bangkalan, Jawa Timur**.

**Polutan yang diambil:** NO2, CO, SO2, CH4

**Rentang waktu:** 24 Agustus 2025 24 Agustus 2026

**Alur kerja notebook ini:**
1. Import pustaka yang dibutuhkan
2. Konfigurasi area of interest (AOI), rentang waktu, dan folder output
3. Autentikasi ke backend openEO Copernicus Data Space (device code flow)
4. Membuat fungsi ekstraksi generik untuk satu polutan
5. Menjalankan ekstraksi untuk keempat polutan (loop) dan menyimpan hasilnya
   sebagai file NetCDF (`.nc`)
6. Mengonversi setiap file NetCDF menjadi file CSV yang siap dipakai pada
   notebook analisis (`2-analisis-bangkalan.ipynb`)

> **Catatan penting:** Backend openEO Sentinel Hub untuk koleksi
> `SENTINEL_5P_L2` hanya mengizinkan **satu band/polutan per request**.
> Karena itu, proses ekstraksi dilakukan dengan cara *loop* satu-per-satu
> untuk setiap polutan, bukan sekaligus dalam satu `load_collection`.


## 1. Import Pustaka

- `openeo` — klien Python resmi untuk berkomunikasi dengan backend openEO.
- `os` — membuat folder output jika belum ada.
- `xarray` — membaca file NetCDF hasil ekstraksi.
- `pandas` — mengonversi data menjadi tabel CSV.
- `time` — jeda kecil antar-request agar tidak membanjiri backend.


In [2]:
import os
import time

import openeo
import pandas as pd
import xarray as xr


## 2. Konfigurasi Umum

Bagian ini berisi semua parameter yang mungkin perlu diubah:
- **URL backend** openEO Copernicus Data Space.
- **Bounding box** dan **polygon** wilayah Kabupaten Bangkalan.
- **Rentang waktu** pengambilan data.
- **Daftar polutan** yang akan diekstraksi beserta nama band-nya di koleksi
  `SENTINEL_5P_L2`.
- **Folder output** untuk file NetCDF dan CSV.

Polygon di bawah adalah persegi sederhana dari bounding box yang diberikan.
Jika kamu punya batas administratif Bangkalan yang lebih presisi (misalnya
dari file GeoJSON BPS/shapefile), silakan ganti variabel `bangkalan_polygon`
dengan koordinat tersebut agar agregasi spasial lebih akurat.


In [3]:
# --- Backend openEO Copernicus Data Space ---
BACKEND_URL = "openeo.dataspace.copernicus.eu"

# --- Bounding box Kabupaten Bangkalan ---
bangkalan_bbox = {
    "west": 112.671,
    "east": 113.116,
    "south": -7.227,
    "north": -6.848,
}

# --- Polygon (GeoJSON) untuk agregasi spasial ---
# Menggunakan bounding box sebagai polygon persegi.
# Urutan koordinat GeoJSON adalah [longitude, latitude], dan ring harus tertutup
# (titik pertama = titik terakhir).
bangkalan_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [bangkalan_bbox["west"], bangkalan_bbox["south"]],
        [bangkalan_bbox["east"], bangkalan_bbox["south"]],
        [bangkalan_bbox["east"], bangkalan_bbox["north"]],
        [bangkalan_bbox["west"], bangkalan_bbox["north"]],
        [bangkalan_bbox["west"], bangkalan_bbox["south"]],
    ]],
}

# --- Rentang waktu ekstraksi ---
START_DATE = "2025-08-24"
END_DATE = "2026-08-24"

# --- Daftar polutan yang akan diekstraksi ---
# key   = nama pendek yang dipakai untuk penamaan file
# value = nama band pada koleksi SENTINEL_5P_L2
POLLUTANTS = {
    "NO2": "NO2",
    "CO": "CO",
    "SO2": "SO2",
    "CH4": "CH4",
}

# --- Folder output ---
NC_DIR = "../data/nc/"
CSV_DIR = "../data/csv/"

os.makedirs(NC_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Konfigurasi siap.")
print("Bounding box Bangkalan:", bangkalan_bbox)
print("Polutan yang akan diekstraksi:", list(POLLUTANTS.keys()))


Konfigurasi siap.
Bounding box Bangkalan: {'west': 112.671, 'east': 113.116, 'south': -7.227, 'north': -6.848}
Polutan yang akan diekstraksi: ['NO2', 'CO', 'SO2', 'CH4']


## 3. Autentikasi ke Copernicus Data Space (Device Code Flow)

`connection.authenticate_oidc()` akan mencoba beberapa metode autentikasi
OpenID Connect secara berurutan, dan jika tidak ada sesi tersimpan, ia akan
otomatis jatuh ke **device code flow**: klien akan mencetak sebuah tautan dan
kode singkat di terminal/output sel.

**Langkah yang perlu kamu lakukan saat sel ini dijalankan:**
1. Buka tautan yang muncul di output (`https://.../device`).
2. Login dengan akun Copernicus Data Space kamu.
3. Masukkan kode yang ditampilkan di output notebook.
4. Tunggu hingga sel selesai — jika berhasil akan muncul pesan konfirmasi
   otentikasi.

Setelah berhasil sekali, token biasanya akan di-cache secara lokal sehingga
kamu tidak perlu login ulang setiap kali menjalankan notebook.


In [4]:
connection = openeo.connect(BACKEND_URL)
connection = connection.authenticate_oidc()

print("Berhasil terhubung dan terautentikasi ke:", BACKEND_URL)


Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=FLGY-AOLR 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.
Berhasil terhubung dan terautentikasi ke: openeo.dataspace.copernicus.eu


## 4. Fungsi Ekstraksi untuk Satu Polutan

Fungsi `extract_pollutant()` melakukan seluruh proses openEO untuk satu
polutan:

1. `load_collection()` — memuat koleksi `SENTINEL_5P_L2`, dibatasi oleh
   bounding box, rentang waktu, dan satu band polutan.
2. `aggregate_temporal_period(period="day", reducer="mean")` — agregasi
   temporal harian (mean), sehingga beberapa lintasan satelit dalam satu hari
   dirata-rata menjadi satu nilai per hari.
3. `aggregate_spatial(geometries=..., reducer="mean")` — agregasi spasial:
   merata-ratakan seluruh piksel di dalam polygon Bangkalan menjadi satu nilai
   per hari (menghasilkan *vector cube* / deret waktu).
4. `save_result(format="netCDF")` — menetapkan format keluaran job sebagai
   NetCDF.
5. Job dijalankan sebagai **batch job** (`create_job` + `start_and_wait`),
   karena ekstraksi setahun data time series biasanya memakan waktu lebih
   lama daripada proses sinkron.
6. Hasil job diunduh ke folder `NC_DIR` dengan nama file
   `bangkalan_<nama_polutan>.nc`.


In [5]:
def extract_pollutant(connection, pollutant_name, band_name,
                       bbox, polygon, start_date, end_date, output_dir):
    """Menarik satu polutan Sentinel-5P L2 untuk area & rentang waktu tertentu.

    Parameters
    ----------
    connection : openeo.Connection
        Koneksi openEO yang sudah terautentikasi.
    pollutant_name : str
        Nama pendek polutan, dipakai untuk penamaan file (mis. "NO2").
    band_name : str
        Nama band pada koleksi SENTINEL_5P_L2 (mis. "NO2").
    bbox : dict
        Bounding box dengan key west/east/south/north.
    polygon : dict
        Geometry GeoJSON (Polygon) untuk agregasi spasial.
    start_date, end_date : str
        Rentang waktu dalam format "YYYY-MM-DD".
    output_dir : str
        Folder tujuan penyimpanan file NetCDF.

    Returns
    -------
    str
        Path file NetCDF hasil ekstraksi.
    """
    print(f"[{pollutant_name}] Membuat data cube ...")
    cube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=[start_date, end_date],
        bands=[band_name],
    )

    print(f"[{pollutant_name}] Agregasi temporal harian (mean) ...")
    cube = cube.aggregate_temporal_period(period="day", reducer="mean")

    print(f"[{pollutant_name}] Agregasi spasial berdasarkan polygon (mean) ...")
    cube = cube.aggregate_spatial(geometries=polygon, reducer="mean")

    result = cube.save_result(format="netCDF")

    print(f"[{pollutant_name}] Mengirim batch job ke backend openEO ...")
    job = result.create_job(title=f"bangkalan_{pollutant_name}")
    job.start_and_wait()

    output_path = os.path.join(output_dir, f"bangkalan_{pollutant_name}.nc")
    job.get_results().download_file(target=output_path)
    print(f"[{pollutant_name}] Selesai -> {output_path}")

    return output_path


## 5. Menjalankan Ekstraksi untuk Semua Polutan

Sel di bawah melakukan *loop* atas kamus `POLLUTANTS` dan memanggil
`extract_pollutant()` untuk setiap polutan (NO2, CO, SO2, CH4). Setiap
polutan diproses sebagai batch job terpisah karena keterbatasan backend
Sentinel Hub yang hanya mengizinkan satu band per request.

`try/except` digunakan agar kegagalan pada satu polutan (misalnya timeout
atau kuota) tidak menghentikan proses ekstraksi polutan lainnya.


In [6]:
nc_paths = {}

for pollutant_name, band_name in POLLUTANTS.items():
    try:
        nc_path = extract_pollutant(
            connection=connection,
            pollutant_name=pollutant_name,
            band_name=band_name,
            bbox=bangkalan_bbox,
            polygon=bangkalan_polygon,
            start_date=START_DATE,
            end_date=END_DATE,
            output_dir=NC_DIR,
        )
        nc_paths[pollutant_name] = nc_path
    except Exception as exc:
        print(f"[{pollutant_name}] GAGAL diekstraksi: {exc}")
    # Jeda singkat agar tidak membanjiri backend dengan request beruntun
    time.sleep(2)

print()
print("Ringkasan file NetCDF yang berhasil dibuat:")
for pollutant_name, path in nc_paths.items():
    print(f"  - {pollutant_name}: {path}")


[NO2] Membuat data cube ...
[NO2] Agregasi temporal harian (mean) ...
[NO2] Agregasi spasial berdasarkan polygon (mean) ...
[NO2] Mengirim batch job ke backend openEO ...
0:00:00 Job 'j-260830124102444e9969ea847ec49bea': send 'start'
0:00:02 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:00:07 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:00:14 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:00:22 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:00:32 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:00:45 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:01:01 Job 'j-260830124102444e9969ea847ec49bea': queued (progress 0%)
0:01:20 Job 'j-260830124102444e9969ea847ec49bea': running (progress N/A)
0:01:44 Job 'j-260830124102444e9969ea847ec49bea': running (progress N/A)
0:02:14 Job 'j-260830124102444e9969ea847ec49bea': running (progress N/A)
0:02:52 Job 'j-260830124102444e9969ea847ec49bea': 

## 6. Konversi NetCDF ke CSV

Fungsi `convert_nc_to_csv()` membuka file NetCDF hasil ekstraksi dengan
`xarray`, mengubahnya menjadi `pandas.DataFrame` (`to_dataframe()`), lalu
merapikan kolomnya (nama tanggal & nilai konsentrasi polutan) sebelum
disimpan sebagai CSV di folder `CSV_DIR`.

Struktur CSV yang dihasilkan untuk setiap polutan kurang lebih:

| date | <nama_polutan> |
|------|----------------|
| 2025-08-24 | 0.000123 |
| 2025-08-25 | 0.000119 |
| ... | ... |


In [7]:
def convert_nc_to_csv(nc_path, pollutant_name, csv_dir):
    """Membuka file NetCDF hasil aggregate_spatial dan menyimpannya sebagai CSV."""
    ds = xr.open_dataset(nc_path)
    df = ds.to_dataframe().reset_index()

    # Nama kolom waktu & nilai bisa berbeda-beda tergantung struktur NetCDF
    # yang dikembalikan backend, sehingga kita cari otomatis kolom yang
    # menyerupai waktu dan kolom nilai numerik polutan.
    time_col_candidates = [c for c in df.columns if "time" in c.lower() or c.lower() == "t"]
    time_col = time_col_candidates[0] if time_col_candidates else df.columns[0]

    value_col_candidates = [
        c for c in df.columns
        if c != time_col and pd.api.types.is_numeric_dtype(df[c])
    ]
    value_col = value_col_candidates[0] if value_col_candidates else df.columns[-1]

    out_df = df[[time_col, value_col]].rename(
        columns={time_col: "date", value_col: pollutant_name}
    )
    out_df["date"] = pd.to_datetime(out_df["date"])
    out_df = out_df.sort_values("date").reset_index(drop=True)

    csv_path = os.path.join(csv_dir, f"bangkalan_{pollutant_name}.csv")
    out_df.to_csv(csv_path, index=False)
    print(f"[{pollutant_name}] CSV disimpan -> {csv_path} ({len(out_df)} baris)")
    return csv_path


csv_paths = {}
for pollutant_name, nc_path in nc_paths.items():
    csv_paths[pollutant_name] = convert_nc_to_csv(nc_path, pollutant_name, CSV_DIR)

print()
print("Ekstraksi & konversi selesai. File CSV siap dipakai di notebook analisis.")


[NO2] CSV disimpan -> ../data/csv/bangkalan_NO2.csv (292 baris)
[CO] CSV disimpan -> ../data/csv/bangkalan_CO.csv (275 baris)
[SO2] CSV disimpan -> ../data/csv/bangkalan_SO2.csv (322 baris)
[CH4] CSV disimpan -> ../data/csv/bangkalan_CH4.csv (82 baris)

Ekstraksi & konversi selesai. File CSV siap dipakai di notebook analisis.


## Selesai

Setelah notebook ini dijalankan, kamu akan memiliki:

- `../data/nc/bangkalan_<POLUTAN>.nc` — data mentah hasil openEO
- `../data/csv/bangkalan_<POLUTAN>.csv` — data deret waktu harian yang rapi

Lanjutkan ke notebook **`2-analisis-bangkalan.ipynb`** untuk melakukan
eksplorasi, visualisasi peta, penanganan *missing values*, dan deteksi
outlier menggunakan `IsolationForest`.
